# Local GPU setup — run the Days 1-3 benchmark without Modal

Mirrors what `Dockerfile.modal` / `modal_app.py` do inside the Modal container, but run directly on this JupyterLab GPU instance. Does not touch `modal_app.py`, `Dockerfile.modal`, or `docs/MODAL.md` — those stay wired up for Modal runs.

Run the cells top to bottom. There are two spots marked **RESTART KERNEL** below — do that manually (Kernel menu → Restart Kernel) before continuing, otherwise the already-running kernel process won't see newly installed packages.

## 1. Clone the repo

In [ ]:
!git clone https://github.com/MaheshGouru/async-vla-latency-bench.git

In [ ]:
%cd ~/async-vla-latency-bench
!git checkout mathew_branch   # or whichever branch you pushed to

## 2. Install the package + pinned LeRobot/LIBERO deps

Use `%pip`, not `!pip` — `%pip` always targets the kernel's own interpreter, which avoids the kernel/shell interpreter-mismatch trap.

In [ ]:
%pip install -e .

In [ ]:
# Pin this to match LEROOT_COMMIT in modal_app.py if you want results comparable to Modal runs.
LEROBOT_COMMIT = "main"
%pip install "lerobot[pi,libero] @ git+https://github.com/huggingface/lerobot.git@{LEROBOT_COMMIT}"

### ⚠️ RESTART KERNEL now

The kernel process was already running before `lerobot`/`libero` were installed, so it won't see them yet. **Kernel menu → Restart Kernel**, then continue from the next cell (you don't need to re-run the cells above).

In [ ]:
# Sanity check after restarting: should print a path with no error.
import libero
print(libero.__file__)

## 3. EGL rendering env vars

Needed for headless MuJoCo rendering on a GPU box (matches `Dockerfile.modal` lines 6-10).

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

## 4. Pre-create `~/.libero/config.yaml`

Avoids an interactive `input()` hang the first time `libero.libero` is imported (matches `Dockerfile.modal` lines 52-68). Only `import libero` (top-level) is safe before this runs — never `import libero.libero`.

In [ ]:
import os
import yaml
import libero

root = os.path.join(os.path.dirname(libero.__file__), "libero")
cfg = {
    "benchmark_root": root,
    "bddl_files": os.path.join(root, "bddl_files"),
    "init_states": os.path.join(root, "init_files"),
    "datasets": os.path.join(root, "../datasets"),
    "assets": os.path.join(root, "assets"),
}

os.makedirs(os.path.expanduser("~/.libero"), exist_ok=True)
with open(os.path.expanduser("~/.libero/config.yaml"), "w") as f:
    yaml.dump(cfg, f)

print(open(os.path.expanduser("~/.libero/config.yaml")).read())

## 5. HuggingFace token

Replaces the Modal Secret `hf-token`. Paste your token below — don't commit this cell's output/notebook with the token filled in.

In [ ]:
import os
os.environ["HF_TOKEN"] = "<your_hf_token>"

## 6. Run the pipeline

Each cell is the local equivalent of one `modal run modal_app.py::main --command <x>` step (see `_run_script` in `modal_app.py`). `output_dir` defaults to the local relative path `async_vla_benchmark/outputs` already set in `async_vla_benchmark/configs/days1_3.yaml`, so no Modal volume is needed. Run in order; later steps depend on earlier ones' outputs.

In [ ]:
CONFIG = "async_vla_benchmark/configs/days1_3.yaml"
OUTPUT_DIR = "async_vla_benchmark/outputs"

In [ ]:
!python -m async_vla_benchmark.scripts.inspect_setup --output-dir {OUTPUT_DIR}

In [ ]:
!python -m async_vla_benchmark.scripts.select_tasks --config {CONFIG} --output-dir {OUTPUT_DIR}

In [ ]:
!python -m async_vla_benchmark.scripts.profile_latency --config {CONFIG} --output-dir {OUTPUT_DIR}

In [ ]:
!python -m async_vla_benchmark.scripts.run_benchmark --config {CONFIG} --output-dir {OUTPUT_DIR} --experiment core

In [ ]:
!python -m async_vla_benchmark.scripts.validate_results --output-dir {OUTPUT_DIR}

In [ ]:
!python -m async_vla_benchmark.scripts.make_figures --output-dir {OUTPUT_DIR}

## Troubleshooting

If an `import` fails right after a `%pip install` in the same session, restart the kernel first before assuming the install failed — the running kernel process caches the package listing from before the install. Verify with:

In [ ]:
import sys, site
print("executable:", sys.executable)
print("user site:", site.getusersitepackages())
print("user site enabled:", site.ENABLE_USER_SITE)
print([p for p in sys.path if "site-packages" in p])